# 📦 Data Validation and Bundle Release

This notebook comprehensively validates the generated synthetic data
and packages it as a **prepared data bundle** for learners.

## Validation Items

| # | Validation | Description |
|---|-----------|-------------|
| 1 | Comprehensive validation | Schema, grounding, conditions, tool call consistency |
| 2 | Split leakage | Scenario family isolation, evaluation contamination check |
| 3 | Backend conversion | Export to LoRA/OSFT-specific formats |
| 4 | Bundle build | Manifest, checksums, metadata |
| 5 | Loader validation | Verify loaders/tokenizers for both backends |
| 6 | Dataset card | Documentation and publication |

## Bundle Output

```
tau-knowledge-v1/
├── manifest.json, checksums.sha256
├── canonical/{train,validation}.jsonl
├── training/{lora,osft}/{train,validation}.jsonl
├── kb/documents.jsonl
├── metadata/{provenance.jsonl, splits.json}
├── reports/{quality.json, quality.md, backend-validation.json}
├── DATASET_CARD.md, LICENSES/
└── configs/{lora.yaml, osft.yaml}
```

In [ ]:
"""Run comprehensive validation."""

import os
import sys
import json
import subprocess
from pathlib import Path
from collections import Counter

from rhoai_model_training_lab.config import (
    load_env, load_yaml_config, load_bundle_config, PROJECT_ROOT,
)

load_env()

sdg_config = load_yaml_config("configs/sdg.yaml")
prep_config = load_yaml_config("configs/data-preparation.yaml")
release_config = load_bundle_config()

canonical_path = PROJECT_ROOT / sdg_config["output"]["canonical_path"]

print("=" * 70)
print("🔍 Comprehensive Data Validation")
print("=" * 70)

validation_results = {
    "schema_valid": 0,
    "schema_invalid": 0,
    "grounding_ok": 0,
    "tool_valid": 0,
    "type_counts": Counter(),
    "errors": [],
}

if canonical_path.exists():
    from rhoai_model_training_lab.schemas.data import CanonicalSample

    all_samples = []
    for cf in sorted(canonical_path.glob("*.jsonl")):
        with open(cf) as f:
            for lineno, line in enumerate(f, 1):
                if not line.strip():
                    continue
                try:
                    data = json.loads(line)
                    sample = CanonicalSample(**data)
                    all_samples.append(sample)
                    validation_results["schema_valid"] += 1
                    validation_results["type_counts"][sample.sample_type.value] += 1

                    # Check tool schema consistency
                    if sample.tools:
                        validation_results["tool_valid"] += 1

                    # Check grounding
                    if sample.source_doc_ids:
                        validation_results["grounding_ok"] += 1

                except Exception as exc:
                    validation_results["schema_invalid"] += 1
                    validation_results["errors"].append(f"{cf.name}:{lineno}: {exc}")

    total = validation_results["schema_valid"] + validation_results["schema_invalid"]
    print(f"  Total records: {total}")
    print(f"  Schema valid: {validation_results['schema_valid']}")
    print(f"  Schema invalid: {validation_results['schema_invalid']}")
    print(f"  With tools: {validation_results['tool_valid']}")
    print(f"  With grounding docs: {validation_results['grounding_ok']}")

    print("\n  Distribution by type:")
    for stype, count in validation_results["type_counts"].most_common():
        pct = count / max(total, 1) * 100
        print(f"    {stype:<25} {count:>5} ({pct:.1f}%)")

    if validation_results["errors"]:
        print(f"\n  Errors ({len(validation_results['errors'])}):") 
        for err in validation_results["errors"][:10]:
            print(f"    ❌ {err}")
else:
    print(f"  ❌ Canonical data path not found: {canonical_path}")
    print("     Please run 02_generate_synthetic.ipynb first.")
    all_samples = []

In [ ]:
"""Check split leakage and contamination."""

print("=" * 70)
print("🔒 Split Leakage and Contamination Check")
print("=" * 70)

if all_samples:
    split_config = prep_config["splits"]

    # Group by scenario family
    family_groups = {}
    for sample in all_samples:
        family = sample.scenario_family or "unknown"
        if family not in family_groups:
            family_groups[family] = []
        family_groups[family].append(sample.sample_id)

    print(f"  Scenario families: {len(family_groups)}")
    print(f"  Split method: {split_config['method']}")
    print(f"  Family isolation: {split_config['family_isolation']}")

    # Check for potential leakage
    print("\n  Family size distribution:")
    sizes = [len(ids) for ids in family_groups.values()]
    print(f"    Mean: {sum(sizes)/len(sizes):.1f}")
    print(f"    Min/Max: {min(sizes)} / {max(sizes)}")

    # Check for duplicate sample IDs
    all_ids = [s.sample_id for s in all_samples]
    unique_ids = set(all_ids)
    duplicates = len(all_ids) - len(unique_ids)
    if duplicates > 0:
        print(f"\n  ❌ Duplicate IDs found: {duplicates}")
    else:
        print(f"\n  ✅ No duplicate IDs")

    # Near-duplicate check (simplified)
    print("\n  Duplicate check:")
    content_hashes = set()
    near_dupes = 0
    for sample in all_samples:
        content = " ".join(m.content or "" for m in sample.messages if m.content)
        import hashlib
        h = hashlib.md5(content.encode()).hexdigest()
        if h in content_hashes:
            near_dupes += 1
        content_hashes.add(h)

    print(f"    Exact duplicates: {near_dupes}")
    print(f"    {'✅' if near_dupes == 0 else '⚠️ '} {'No duplicates' if near_dupes == 0 else 'Deduplication required'}")

    # Contamination check note
    print("\n  Contamination check:")
    print("    ℹ️  Contamination check against evaluation tasks must run in an isolated process.")
    print("    Only minimal pass/fail and hash-based results are returned.")
    print("    Manual execution: python scripts/validate_synthetic.py --contamination-check")
else:
    print("  ⚠️  No data to check.")

In [ ]:
"""Export to backend-specific formats."""

print("=" * 70)
print("📤 Export to Backend-Specific Formats")
print("=" * 70)

bundle_base = PROJECT_ROOT / release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")

if all_samples:
    # Split samples
    train_ratio = prep_config["splits"]["train_ratio"]
    seed = prep_config["splits"]["seed"]

    import random
    rng = random.Random(seed)

    # Group by family and assign splits
    families = list(family_groups.keys())
    rng.shuffle(families)

    split_point = int(len(families) * train_ratio)
    train_families = set(families[:split_point])
    val_families = set(families[split_point:])

    train_samples = [s for s in all_samples if (s.scenario_family or "unknown") in train_families]
    val_samples = [s for s in all_samples if (s.scenario_family or "unknown") in val_families]

    print(f"  Train families: {len(train_families)}, Validation families: {len(val_families)}")
    print(f"  Train samples: {len(train_samples)}, Validation samples: {len(val_samples)}")

    # Export canonical
    for split_name, samples in [("train", train_samples), ("validation", val_samples)]:
        out_dir = bundle_base / "canonical"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_file = out_dir / f"{split_name}.jsonl"
        with open(out_file, "w") as f:
            for s in samples:
                f.write(s.model_dump_json() + "\n")
        print(f"  ✅ {out_file}: {len(samples)} samples")

    # Export backend-specific (LoRA and OSFT)
    for backend in ["lora", "osft"]:
        for split_name, samples in [("train", train_samples), ("validation", val_samples)]:
            out_dir = bundle_base / "training" / backend
            out_dir.mkdir(parents=True, exist_ok=True)
            out_file = out_dir / f"{split_name}.jsonl"

            with open(out_file, "w") as f:
                for s in samples:
                    # Convert to backend format (messages only, no provenance)
                    record = {
                        "messages": [m.model_dump(exclude_none=True) for m in s.messages],
                    }
                    if s.tools:
                        record["tools"] = [t.model_dump() for t in s.tools]
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
            print(f"  ✅ {out_file}: {len(samples)} samples")

    print("\n  ℹ️  LoRA/OSFT data are converted from the same canonical samples.")
    print("     Loss masking: only assistant responses are training targets")
else:
    print("  ⚠️  No data to export.")

In [ ]:
"""Build bundle with manifests and checksums."""

from rhoai_model_training_lab.data import compute_file_checksum
from rhoai_model_training_lab.schemas.data import BundleManifest

print("=" * 70)
print("📦 Bundle Build")
print("=" * 70)

if all_samples:
    model_id = release_config["model_profile"]["model_id"]
    model_revision = release_config["model_profile"]["model_revision"]

    # Build manifest
    type_dist = dict(validation_results["type_counts"])

    manifest = BundleManifest(
        bundle_name=release_config["bundle"]["name"],
        bundle_version=release_config["bundle"]["version"],
        tau_version=os.environ.get("TAU_BENCH_VERSION", ""),
        tau_commit_sha=os.environ.get("TAU_BENCH_COMMIT_SHA", ""),
        model_id=model_id,
        model_revision=model_revision,
        tokenizer_id=release_config["model_profile"]["tokenizer_id"],
        canonical_train_count=len(train_samples),
        canonical_validation_count=len(val_samples),
        split_policy=prep_config["splits"]["method"],
        sample_type_distribution=type_dist,
    )

    # Save manifest
    manifest_path = bundle_base / "manifest.json"
    with open(manifest_path, "w") as f:
        f.write(manifest.model_dump_json(indent=2))
    print(f"  ✅ Manifest: {manifest_path}")

    # Compute and save checksums
    checksum_path = bundle_base / "checksums.sha256"
    with open(checksum_path, "w") as f:
        for file_path in sorted(bundle_base.rglob("*")):
            if file_path.is_file() and file_path.name != "checksums.sha256":
                rel = file_path.relative_to(bundle_base)
                h = compute_file_checksum(file_path)
                f.write(f"{h}  {rel}\n")
    print(f"  ✅ Checksums: {checksum_path}")

    # Save split metadata
    from rhoai_model_training_lab.schemas.data import SplitInfo
    split_info = SplitInfo(
        method=prep_config["splits"]["method"],
        seed=prep_config["splits"]["seed"],
        train_ids=[s.sample_id for s in train_samples],
        validation_ids=[s.sample_id for s in val_samples],
        train_count=len(train_samples),
        validation_count=len(val_samples),
        family_partition={
            f: "train" if f in train_families else "validation"
            for f in families
        },
    )
    meta_dir = bundle_base / "metadata"
    meta_dir.mkdir(parents=True, exist_ok=True)
    with open(meta_dir / "splits.json", "w") as f:
        f.write(split_info.model_dump_json(indent=2))
    print(f"  ✅ Split metadata: {meta_dir / 'splits.json'}")

    print(f"\n  Bundle path: {bundle_base}")
else:
    print("  ⚠️  No data to build.")

In [ ]:
"""Validate with both backend loaders."""

from rhoai_model_training_lab.data import BundleManager, validate_prepared_dataset

print("=" * 70)
print("🔍 Backend Loader Validation")
print("=" * 70)

if bundle_base.exists() and (bundle_base / "manifest.json").exists():
    # Full validation
    validation = validate_prepared_dataset(bundle_base)

    if validation["valid"]:
        print("✅ Bundle validation passed")
    else:
        print("❌ Bundle validation failed")
        for err in validation["errors"]:
            print(f"  ❌ {err}")

    if validation["warnings"]:
        for warn in validation["warnings"]:
            print(f"  ⚠️  {warn}")

    # Test loading with both backends
    mgr = BundleManager.load_bundle(bundle_base)

    for backend in ["lora", "osft"]:
        print(f"\n  --- {backend.upper()} loader test ---")
        try:
            train = mgr.get_training_samples(backend, "train")
            val = mgr.get_training_samples(backend, "validation")
            print(f"    Train: {len(train)} samples ✅")
            print(f"    Validation: {len(val)} samples ✅")

            # Verify message format
            if train:
                first = train[0]
                assert "messages" in first, "messages field missing"
                assert len(first["messages"]) > 0, "empty message list"
                print(f"    Message format ✅")
        except Exception as exc:
            print(f"    ❌ Failed: {exc}")

    # Tokenizer check
    model_id = release_config["model_profile"]["model_id"]
    compat = mgr.validate_compatibility(model_id)
    print(f"\n  Model compatibility: {'✅' if not compat.errors else '❌'}")
    for err in compat.errors:
        print(f"    ❌ {err}")
else:
    print("  ⚠️  Bundle does not exist. Please build it first.")

In [ ]:
"""Generate dataset card and publish."""

print("=" * 70)
print("📄 Dataset Card Generation and Publication")
print("=" * 70)

if bundle_base.exists() and (bundle_base / "manifest.json").exists():
    # Generate dataset card
    dataset_card = f"""# τ-Knowledge Banking Dataset Bundle

## Overview

- **Name**: {release_config['bundle']['name']}
- **Version**: {release_config['bundle']['version']}
- **Domain**: τ-Knowledge banking_knowledge
- **Model Target**: {release_config['model_profile']['model_id']}

## Contents

- Canonical training and validation samples
- LoRA and OSFT backend-specific exports
- KB document snapshot
- Provenance and split metadata
- Quality reports

## Generation Method

Synthetic data generated using sdg_hub with independent validation.
See `reports/quality.json` for detailed acceptance/rejection statistics.

## Validation Method

- Schema validation (Pydantic models)
- Grounding check against KB documents
- Tool schema consistency check
- Scenario family split isolation
- Evaluation contamination check (isolated process)

## Split Methodology

Scenario family-based splitting ensures that paraphrases,
name/number substitutions, and sibling examples remain in the same split.

## Limitations

- Synthetic data may not cover all banking scenarios
- Quality depends on teacher model capability
- This is a kb_adaptation experiment, not a training-free protocol
- Small model (4B) may not achieve high agent task performance

## Redistribution

Check LICENSES/ directory for applicable conditions.
"""

    card_path = bundle_base / "DATASET_CARD.md"
    with open(card_path, "w") as f:
        f.write(dataset_card)
    print(f"✅ Dataset card: {card_path}")

    # Create licenses directory
    licenses_dir = bundle_base / "LICENSES"
    licenses_dir.mkdir(exist_ok=True)
    print(f"✅ Licenses directory: {licenses_dir}")

    # Publish options
    print("\n--- Publish Options ---")
    publish_targets = release_config.get("publish", {}).get("targets", [])
    for target in publish_targets:
        target_type = target.get("type", "")
        if target_type == "s3":
            bucket = target.get("bucket", "")
            prefix = target.get("prefix", "")
            print(f"  📤 S3: s3://{bucket}/{prefix}")
            print(f"     aws s3 sync {bundle_base} s3://{bucket}/{prefix}{release_config['bundle']['name']}-{release_config['bundle']['version']}/")
        elif target_type == "local":
            local_path = target.get("path", "")
            print(f"  💾 Local: {local_path}")

    # Build script reference
    print("\n  Or use the build script:")
    print(f"    python scripts/build_prepared_bundle.py --config configs/data-release.yaml")

    # Final summary
    print("\n" + "=" * 70)
    print("🎉 Bundle preparation complete!")
    print("=" * 70)
    print(f"  Bundle path: {bundle_base}")
    if all_samples:
        print(f"  Train samples: {len(train_samples)}")
        print(f"  Validation samples: {len(val_samples)}")
    print(f"  Target model: {release_config['model_profile']['model_id']}")
    print("\n  Learners can download this bundle and start training immediately.")
    print("  Training notebooks: notebooks/03_lora_finetuning.ipynb, 04_osft_finetuning.ipynb")
else:
    print("  ⚠️  Bundle does not exist. Please build it first.")